In [1]:
!pip -q install datasets

import os
import re
import zipfile
import urllib.request
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42)
np.random.seed(42)

In [2]:
dataset = load_dataset("ag_news")

label_names = dataset["train"].features["label"].names
num_classes = len(label_names)

train_limit = 40000
test_limit = None

train_data = dataset["train"]
test_data = dataset["test"]

if train_limit is not None:
    train_data = train_data.shuffle(seed=42).select(range(train_limit))

if test_limit is not None:
    test_data = test_data.select(range(test_limit))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text.lower())


vocab_size = 30000
max_length = 80
embedding_dim = 100

counter = Counter()
for example in train_data:
    counter.update(tokenize(example["text"]))

word2idx = {"<PAD>": 0, "<UNK>": 1}
idx2word = {0: "<PAD>", 1: "<UNK>"}

for idx, (word, _) in enumerate(counter.most_common(vocab_size - 2), start=2):
    word2idx[word] = idx
    idx2word[idx] = word

In [4]:
def download_glove():
    glove_zip_path = "glove.6B.zip"
    glove_txt_path = "glove.6B.100d.txt"
    glove_url = "http://nlp.stanford.edu/data/glove.6B.zip"

    if not os.path.exists(glove_txt_path):
        if not os.path.exists(glove_zip_path):
            urllib.request.urlretrieve(glove_url, glove_zip_path)

        with zipfile.ZipFile(glove_zip_path, "r") as zip_ref:
            zip_ref.extract(glove_txt_path)

    return glove_txt_path


glove_path = download_glove()
glove_embeddings = {}

with open(glove_path, "r", encoding="utf8") as file:
    for line in file:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_embeddings[word] = vector

embedding_matrix = np.zeros((len(word2idx), embedding_dim), dtype=np.float32)
embedding_matrix[1] = np.random.normal(scale=0.6, size=(embedding_dim,))

for word, idx in word2idx.items():
    if word in glove_embeddings:
        embedding_matrix[idx] = glove_embeddings[word]
    elif idx != 0 and idx != 1:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

KeyboardInterrupt: 

In [ ]:
def encode(text):
    token_ids = [word2idx.get(token, word2idx["<UNK>"]) for token in tokenize(text)]
    token_ids = token_ids[:max_length]
    padding = [word2idx["<PAD>"]] * (max_length - len(token_ids))
    return token_ids + padding


class AGNewsDataset(Dataset):
    def __init__(self, split):
        self.data = split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]
        x = torch.tensor(encode(text), dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)
        return x, y


batch_size = 256

train_dataset = AGNewsDataset(train_data)
test_dataset = AGNewsDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
class AGNewsPretrainedNet(nn.Module):
    def __init__(self, embedding_matrix, num_classes, freeze_embeddings=True):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
        self.embedding.weight.requires_grad = not freeze_embeddings
        self.fc1 = nn.Linear(embedding_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        embeddings = self.embedding(x)
        mask = (x != 0).unsqueeze(-1)
        summed = (embeddings * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        x = summed / lengths
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        return self.fc3(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AGNewsPretrainedNet(
    embedding_matrix=embedding_matrix,
    num_classes=num_classes,
    freeze_embeddings=True,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda parameter: parameter.requires_grad, model.parameters()), lr=0.001)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


epochs = 5

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
    )
    print(f"Epoch {epoch + 1}/{epochs} | Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")

In [ ]:
def evaluate(model, loader, device, num_classes):
    model.eval()
    confusion = np.zeros((num_classes, num_classes), dtype=int)

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            logits = model(inputs)
            predictions = torch.argmax(logits, dim=1)

            for gold, pred in zip(labels.cpu().numpy(), predictions.cpu().numpy()):
                confusion[gold, pred] += 1

    accuracy = np.trace(confusion) / np.sum(confusion)
    return confusion, accuracy


confusion, test_accuracy = evaluate(model, test_loader, device, num_classes)

print("Accuracy en test:", round(test_accuracy, 4))
print("Matriz de confusión")
print(confusion)

print("Métricas por clase")
for i, class_name in enumerate(label_names):
    tp = confusion[i, i]
    fp = confusion[:, i].sum() - tp
    fn = confusion[i, :].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    print(f"{class_name:8s} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

In [ ]:
def predict_text(model, text, device):
    model.eval()
    x = torch.tensor(encode(text), dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
        predicted_idx = int(np.argmax(probabilities))

    return label_names[predicted_idx], probabilities


examples = [
    "The government announced new international trade measures after talks with European leaders.",
    "The team won the championship after scoring two goals in the final minutes.",
    "The company reported higher quarterly revenue as stock markets reacted positively.",
    "Researchers introduced a new chip design for faster artificial intelligence computing.",
]

for text in examples:
    predicted_class, probabilities = predict_text(model, text, device)
    print("Texto:", text)
    print("Predicción:", predicted_class)
    print("Probabilidades:", {label_names[i]: round(float(p), 4) for i, p in enumerate(probabilities)})